공통설정

In [2]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.callbacks import UsageMetadataCallbackHandler

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]
LOWER = os.environ["OPENAI_LOWER_MODEL"]
JUDGE = os.environ["OPENAI_JUDGE_MODEL"]
EMBEDDING = os.environ["OPENAI_EMBEDDING_MODEL"]

PRICES = {MODEL: (2.50, 15.00), LOWER: (0.75, 4.50), EMBEDDING: (0.02, 0.0)}  # 1M 토큰당 (입력, 출력) USD
usage = UsageMetadataCallbackHandler()

llm = ChatOpenAI(model=MODEL)
embeddings = OpenAIEmbeddings(model=EMBEDDING)
PDF_DIR = "./manual"
GOLDEN = "./golden/manual.jsonl"

## ingestion (pdf 로더)

In [5]:
import re

# 마크다운 개수
def count_markdown_tables(text):
    pattern = r'\|.*\|\s*\n\|(?:\s*:?-+:?\s*\|)+'
    return len(re.findall(pattern, text))

In [ ]:
# 로더 비교
import time
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PDFPlumberLoader, PyMuPDFLoader

t0 = time.perf_counter()

# 파일 1 : 텍스트 + 사진 8페이지, 표 13페이지, 사진 15페이지
ven_total_manual = PDF_LIST[0]

for name, Loader in [("PyPDFLoader", PyPDFLoader),
                     ("PDFPlumberLoader", PDFPlumberLoader),
                     ("PyMuPDFLoader", PyMuPDFLoader)
                     ]:
    docs = Loader(ven_total_manual).load()
    print(f"{name:18} {len(docs):3d}쪽  {time.perf_counter() - t0:5.2f}s")
    print(f"텍스트+사진 : {len(docs[7].page_content):3d}자 메타데이터 {len(docs[7].metadata)}개\n", repr(docs[7].page_content))
    print(f"표 : {len(docs[12].page_content):3d}자 메타데이터 {len(docs[12].metadata)}개\n", repr(docs[12].page_content))
    print(f"사진 : {len(docs[14].page_content):3d}자 메타데이터 {len(docs[14].metadata)}개\n", repr(docs[14].page_content))
    print("\n")

markdown_PyMuPDFLoader = PyMuPDFLoader(
    ven_total_manual,
    mode="page",
    extract_tables="markdown"
)    
docs = markdown_PyMuPDFLoader.load()
name = "markdown_PyMuPDFLoader"
print(f"{name:18} {len(docs):3d}쪽  {time.perf_counter() - t0:5.2f}s")
print(f"텍스트+사진 : {len(docs[7].page_content):3d}자 메타데이터 {len(docs[7].metadata)}개\n", repr(docs[7].page_content))
print(f"표 : {len(docs[12].page_content):3d}자 메타데이터 {len(docs[12].metadata)}개\n", repr(docs[12].page_content))
display(Markdown(docs[12].page_content))
print(f"사진 : {len(docs[14].page_content):3d}자 메타데이터 {len(docs[14].metadata)}개\n", repr(docs[14].page_content))
print("\n")
print(
    "표가 있는 페이지:",
    sum(count_markdown_tables(doc.page_content) > 0 for doc in docs),
    "/",
    len(docs)
)
print(
    "전체 표 개수:",
    sum(count_markdown_tables(doc.page_content) for doc in docs)
)


In [6]:
import re
import time
import pandas as pd
import pymupdf

from pathlib import Path
from IPython.display import display, Markdown
from langchain_community.document_loaders import (
    PyPDFLoader,
    PDFPlumberLoader,
    PyMuPDFLoader,
)


def benchmark_pdf_loaders(
    pdf_path,
    text_page=None,
    table_page=None,
    image_page=None,
):
    """
    여러 PDF Loader의 추출 결과를 비교한다.

    페이지 번호는 1부터 시작한다.

    text_page / table_page / image_page가
    - None이거나
    - 1보다 작거나
    - 전체 페이지 수보다 크면

    해당 항목은 비교하지 않고 건너뛴다.
    """

    pdf_path = Path(pdf_path)

    # 실제 PDF 페이지 수
    with pymupdf.open(pdf_path) as pdf:
        actual_pages = pdf.page_count


    # -------------------------------------------
    # 유효한 페이지인지 확인
    # -------------------------------------------
    def is_valid_page(page_num):
        return (
            isinstance(page_num, int)
            and 1 <= page_num <= actual_pages
        )


    loaders = {
        "PyPDFLoader": {
            "loader": lambda: PyPDFLoader(
                str(pdf_path)
            ).load(),
            "markdown": False,
        },

        "PDFPlumberLoader": {
            "loader": lambda: PDFPlumberLoader(
                str(pdf_path)
            ).load(),
            "markdown": False,
        },

        "PyMuPDFLoader": {
            "loader": lambda: PyMuPDFLoader(
                str(pdf_path),
                mode="page",
            ).load(),
            "markdown": False,
        },

        "PyMuPDF + Markdown": {
            "loader": lambda: PyMuPDFLoader(
                str(pdf_path),
                mode="page",
                extract_tables="markdown",
            ).load(),
            "markdown": True,
        },
    }


    results = []


    for loader_name, config in loaders.items():

        try:

            # -------------------------------------------
            # Loader 실행 시간 측정
            # -------------------------------------------
            start = time.perf_counter()

            docs = config["loader"]()

            elapsed = time.perf_counter() - start


            # -------------------------------------------
            # 선택 페이지
            # 유효하지 않으면 None
            # -------------------------------------------
            text_doc = (
                docs[text_page - 1]
                if is_valid_page(text_page)
                else None
            )

            table_doc = (
                docs[table_page - 1]
                if is_valid_page(table_page)
                else None
            )

            image_doc = (
                docs[image_page - 1]
                if is_valid_page(image_page)
                else None
            )


            # -------------------------------------------
            # 전체 문자 수
            # -------------------------------------------
            total_chars = sum(
                len(doc.page_content)
                for doc in docs
            )


            # -------------------------------------------
            # Markdown 표 통계
            # -------------------------------------------
            if config["markdown"]:

                table_counts = [
                    count_markdown_tables(doc.page_content)
                    for doc in docs
                ]

                table_page_count = sum(
                    count > 0
                    for count in table_counts
                )

                total_table_count = sum(table_counts)

                table_page_ratio = (
                    table_page_count / len(docs)
                    if docs
                    else 0
                )

            else:

                table_page_count = None
                total_table_count = None
                table_page_ratio = None


            # -------------------------------------------
            # 결과 저장
            # -------------------------------------------
            results.append({
                "Loader": loader_name,

                "실행시간(sec)": round(elapsed, 3),

                "전체페이지": len(docs),

                "전체문자수": total_chars,

                "텍스트페이지":
                    text_page if text_doc else None,

                "텍스트페이지_문자수":
                    len(text_doc.page_content)
                    if text_doc else None,

                "표페이지":
                    table_page if table_doc else None,

                "표페이지_문자수":
                    len(table_doc.page_content)
                    if table_doc else None,

                "이미지페이지":
                    image_page if image_doc else None,

                "이미지페이지_문자수":
                    len(image_doc.page_content)
                    if image_doc else None,

                "메타데이터수":
                    len(docs[0].metadata)
                    if docs else 0,

                "Markdown표_페이지수":
                    table_page_count,

                "Markdown표_페이지비율":
                    table_page_ratio,

                "Markdown표_전체개수":
                    total_table_count,

                "상태": "OK",
            })


        except Exception as e:

            results.append({
                "Loader": loader_name,

                "실행시간(sec)": None,

                "전체페이지": None,

                "전체문자수": None,

                "텍스트페이지": None,
                "텍스트페이지_문자수": None,

                "표페이지": None,
                "표페이지_문자수": None,

                "이미지페이지": None,
                "이미지페이지_문자수": None,

                "메타데이터수": None,

                "Markdown표_페이지수": None,
                "Markdown표_페이지비율": None,
                "Markdown표_전체개수": None,

                "상태": f"ERROR: {e}",
            })


    result_df = pd.DataFrame(results)


    # -------------------------------------------
    # Markdown 표 비율 %
    # -------------------------------------------
    result_df["Markdown표_페이지비율"] = (
        result_df["Markdown표_페이지비율"]
        .apply(
            lambda x:
                f"{x:.1%}"
                if pd.notna(x)
                else "-"
        )
    )


    return result_df

In [7]:
# 특정 페이지 컨텐트 내용 보기
def show_page_content(
    pdf_path,
    page_num,
    loader_names=None,
    render_markdown=True
):
    """
    특정 PDF의 특정 페이지를 Loader별로 비교 출력한다.

    page_num은 1부터 시작.
    """

    loaders = {
        "PyPDFLoader": lambda: PyPDFLoader(
            str(pdf_path)
        ).load(),

        "PDFPlumberLoader": lambda: PDFPlumberLoader(
            str(pdf_path)
        ).load(),

        "PyMuPDFLoader": lambda: PyMuPDFLoader(
            str(pdf_path),
            mode="page"
        ).load(),

        "PyMuPDF + Markdown": lambda: PyMuPDFLoader(
            str(pdf_path),
            mode="page",
            extract_tables="markdown"
        ).load(),
    }

    # 지정하지 않으면 전체 Loader 출력
    if loader_names is None:
        loader_names = list(loaders.keys())

    for loader_name in loader_names:

        docs = loaders[loader_name]()

        # 잘못된 페이지면 건너뜀
        if page_num < 1 or page_num > len(docs):
            print(
                f"{loader_name}: "
                f"{page_num}페이지 없음 "
                f"(전체 {len(docs)}페이지)"
            )
            continue

        doc = docs[page_num - 1]

        display(
            Markdown(
                f"{loader_name} "
                f"- {page_num}페이지"
            )
        )

        print(f"문자수: {len(doc.page_content)}")
        print(f"metadata: {doc.metadata}")
        print(f"page_content: {doc.page_content}")

        if render_markdown:
            display(Markdown(doc.page_content))

        print("-" * 100)


In [8]:
# 마크다운 표 확인
def extract_markdown_tables(text: str):
    """
    Markdown 표만 추출해서 리스트로 반환
    """

    pattern = re.compile(
        r"""
        (
            ^\s*\|.*\|\s*$\n
            ^\s*\|?\s*:?-{3,}:?\s*
            (?:\|\s*:?-{3,}:?\s*)+
            \|?\s*$\n
            (?:^\s*\|.*\|\s*$\n?)*
        )
        """,
        re.MULTILINE | re.VERBOSE
    )

    return [
        match.group(1).strip()
        for match in pattern.finditer(text)
    ]

def extract_all_markdown_tables(pdf_path):

    results = []
    docs = PyMuPDFLoader(
    pdf_path,
    mode="page",
    extract_tables="markdown"
    ).load()

    for page_idx, doc in enumerate(docs):

        tables = extract_markdown_tables(
            doc.page_content
        )

        for table_idx, table in enumerate(
            tables,
            start=1
        ):

            results.append({
                "page": page_idx + 1,
                "table_no": table_idx,
                "markdown": table,
            })
            display(Markdown(table))

    return results

In [9]:
PDF_LIST = sorted(Path(PDF_DIR).glob("*.pdf"))
page_configs = [
    [8, 13, 15],     # 1번 PDF
    [4, 27, 25],    # 2번 PDF
    [8, None, 16],   # 3번 PDF
    [None, 23, 9],     # 4번 PDF
    [8, None, 10],
    [6, 13, 17],
    [6, 5, 4],
    [37, 10, 4]
]

for pdf_path, pages in zip(PDF_LIST, page_configs):
    text_page, table_page, image_page = pages

    result_df = benchmark_pdf_loaders(
        pdf_path=pdf_path,
        text_page=text_page,
        table_page=table_page,
        image_page=image_page,
    )

    print(pdf_path)
    display(result_df)


Consider using the pymupdf_layout package for a greatly improved page layout analysis.
manual/1. 협력사 운영 통합 안내서 (p.23).pdf


,Loader,실행시간(sec),전체페이지,전체문자수,텍스트페이지,텍스트페이지_문자수,표페이지,표페이지_문자수,이미지페이지,이미지페이지_문자수,메타데이터수,Markdown표_페이지수,Markdown표_페이지비율,Markdown표_전체개수,상태
0,PyPDFLoader,2.032,23,8366,8,371,13,517,15,408,11,NaN,-,NaN,OK
1,PDFPlumberLoader,3.180,23,8335,8,364,13,518,15,408,11,NaN,-,NaN,OK
2,PyMuPDFLoader,0.129,23,8224,8,357,13,517,15,405,16,NaN,-,NaN,OK
3,PyMuPDF + Markdown,0.998,23,14294,8,357,13,1145,15,405,16,14.0,60.9%,21.0,OK


manual/2. Hmall 소개서 (p.30).pdf


,Loader,실행시간(sec),전체페이지,전체문자수,텍스트페이지,텍스트페이지_문자수,표페이지,표페이지_문자수,이미지페이지,이미지페이지_문자수,메타데이터수,Markdown표_페이지수,Markdown표_페이지비율,Markdown표_전체개수,상태
0,PyPDFLoader,2.549,30,7237,4,312,27,22,25,193,11,NaN,-,NaN,OK
1,PDFPlumberLoader,4.306,30,6783,4,312,27,23,25,170,11,NaN,-,NaN,OK
2,PyMuPDFLoader,0.103,30,6206,4,271,27,20,25,165,16,NaN,-,NaN,OK
3,PyMuPDF + Markdown,2.235,30,8652,4,271,27,1040,25,165,16,6.0,20.0%,11.0,OK


manual/3. 쇼라 소개서 (p.22).pdf


...

,Loader,실행시간(sec),전체페이지,전체문자수,텍스트페이지,텍스트페이지_문자수,표페이지,표페이지_문자수,이미지페이지,이미지페이지_문자수,메타데이터수,Markdown표_페이지수,Markdown표_페이지비율,Markdown표_전체개수,상태
0,PyPDFLoader,7.883,21,6578,6,556,13,361,17,372,11,NaN,-,NaN,OK
1,PDFPlumberLoader,16.597,21,6578,6,558,13,363,17,376,11,NaN,-,NaN,OK
2,PyMuPDFLoader,0.179,21,6473,6,553,13,359,17,370,16,NaN,-,NaN,OK
3,PyMuPDF + Markdown,4.037,21,6986,6,553,13,642,17,370,16,2.0,9.5%,2.0,OK


manual/7. 데이터 영역 광고 제안서 (p.8).pdf


,Loader,실행시간(sec),전체페이지,전체문자수,텍스트페이지,텍스트페이지_문자수,표페이지,표페이지_문자수,이미지페이지,이미지페이지_문자수,메타데이터수,Markdown표_페이지수,Markdown표_페이지비율,Markdown표_전체개수,상태
0,PyPDFLoader,0.115,8,1089,6,353,5,110,4,49,11,NaN,-,NaN,OK
1,PDFPlumberLoader,0.213,8,1108,6,354,5,113,4,50,11,NaN,-,NaN,OK
2,PyMuPDFLoader,0.020,8,994,6,325,5,103,4,46,16,NaN,-,NaN,OK
3,PyMuPDF + Markdown,0.159,8,1748,6,601,5,253,4,46,16,4.0,50.0%,4.0,OK


manual/8. QA가이드 (p.21).pdf


,Loader,실행시간(sec),전체페이지,전체문자수,텍스트페이지,텍스트페이지_문자수,표페이지,표페이지_문자수,이미지페이지,이미지페이지_문자수,메타데이터수,Markdown표_페이지수,Markdown표_페이지비율,Markdown표_전체개수,상태
0,PyPDFLoader,0.974,38,23965,37,304,10,1071,4,199,10,NaN,-,NaN,OK
1,PDFPlumberLoader,1.752,38,23886,37,318,10,1112,4,206,9,NaN,-,NaN,OK
2,PyMuPDFLoader,0.051,38,22239,37,263,10,946,4,179,16,NaN,-,NaN,OK
3,PyMuPDF + Markdown,2.924,38,46743,37,263,10,1883,4,179,16,24.0,63.2%,30.0,OK


PDF별 결과 정리

In [10]:
#1. 협력사 운영 통합 안내서 (p.23).pdf : PyMu + Markdown / 3~22페이지
show_page_content(pdf_path=PDF_LIST[0], page_num=13)

PyPDFLoader - 13페이지

문자수: 517
metadata: {'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-07-03T15:15:27+09:00', 'title': 'PowerPoint 프레젠테이션', 'author': '박시현 선임', 'moddate': '2024-07-03T15:15:27+09:00', 'fasoo_trace_id': 'eyAibm9kZUNvdW50IjogNDYsICJub2RlMSIgOiB7ImRzZCI6IjAxMDAwMDAwMDAwMDE4OTQiLCJsb2dUaW1lIjoiMjAyNC0wNy0wM1QwNTo1NzowOFoiLCJwSUQiOjEsInRyYWNlSWQiOiI0NUYxREM5RDU4OEE0QzVDOTM2NDc3QkNBNkZENTMxRCIsInVzZXJDb2RlIjoiMjMwNjU4NCJ9LCJub2RlMiIgOiB7ImRzZCI6IjAxMDAwMDAwMDAwMDE4OTQiLCJsb2dUaW1lIjoiMjAyNC0wNy0wM1QwNjowMDowMFoiLCJwSUQiOjEsInRyYWNlSWQiOiJBMkZDOUFBNkQ2QTc0MjQ5QTBCNTE2RDYyMTA1ODdGRSIsInVzZXJDb2RlIjoiMjMwNjU4NCJ9LCJub2RlMyIgOiB7ImRzZCI6IjAxMDAwMDAwMDAwMDE4OTQiLCJsb2dUaW1lIjoiMjAyNC0wNy0wM1QwNjowODowOVoiLCJwSUQiOjEsInRyYWNlSWQiOiI4OEI2Rjg1RjE5MDg0OTk3ODQzQkU3Q0YxQUVGNzkxNSIsInVzZXJDb2RlIjoiMjMwNjU4NCJ9LCJub2RlNCIgOiB7ImRzZCI6IjAxMDAwMDAwMDAwMDE4OTQiLCJsb2dUaW1lIjoiMjAyNC0wNy0wM1QwNjoxNToyNFoiLCJwSUQiOjEsInRyYWNlSWQiOiJDNzJEMUU0QzYxN0U0RUVCQ

7. QA
○단계별QA 종류
구분 내용
사전QA
상품별서류및샘플점검을통한상품전반의품질관리업무를총괄
기준일 의뢰: 방송일기준최소14일전의뢰/ 완료: 방송일기준최소4일전완료
업무
법적준수사항이행여부확인, 품질서류검토,
원산지검증, 방송품질소구관련증빙자료검토등
현장QA
제조사, 물류센터방문을통한완제품의진정성을확인
기준일 의뢰: 방송일기준최소10일전의뢰/ 완료: 방송일기준최소4일전완료
업무
제조사위생점검및원료및자재관리점검, 품질표시정보확인,
완제품규격및포장상태점검, 외관불량및구성품혼입점검등
입고QA
현대물류센터입고상품대상, 샘플링점검을통한입고가능여부최종검사
기준일 의뢰: 방송일기준최소3일전의뢰
업무 구성품혼입, 상품오염, 내외관불량표시, 정보오류점검등
○QA관련파트너시스템주요화면
QA 자료관리(VD036)
QA 의뢰및승인현황을확인하는화면입니다.
①의뢰상품의QA관리코드가등록되면,
품질서류파일을업로드할수있습니다.
②파일별진행상태확인이가능합니다.
(미등록/ 등록/ 수정요청/ 승인완료)
③‘수정요청’ 시파일재업로드가필요합니다.

----------------------------------------------------------------------------------------------------


PDFPlumberLoader - 13페이지

...

----------------------------------------------------------------------------------------------------


PyMuPDF + Markdown - 13페이지

문자수: 1145
metadata: {'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-07-03T15:15:27+09:00', 'source': 'manual/1. 협력사 운영 통합 안내서 (p.23).pdf', 'file_path': 'manual/1. 협력사 운영 통합 안내서 (p.23).pdf', 'total_pages': 23, 'format': 'PDF 1.5', 'title': 'PowerPoint 프레젠테이션', 'author': '박시현 선임', 'subject': '', 'keywords': '', 'moddate': '2024-07-03T15:15:27+09:00', 'trapped': '', 'modDate': "D:20240703151527+09'00'", 'creationDate': "D:20240703151527+09'00'", 'page': 12}
page_content: 7. QA
○단계별QA 종류
구분
내용
사전QA
상품별서류및샘플점검을통한상품전반의품질관리업무를총괄
기준일
의뢰: 방송일기준최소14일전의뢰/ 완료: 방송일기준최소4일전완료
업무
법적준수사항이행여부확인, 품질서류검토,
원산지검증, 방송품질소구관련증빙자료검토등
현장QA
제조사, 물류센터방문을통한완제품의진정성을확인
기준일
의뢰: 방송일기준최소10일전의뢰/ 완료: 방송일기준최소4일전완료
업무
제조사위생점검및원료및자재관리점검, 품질표시정보확인,
완제품규격및포장상태점검, 외관불량및구성품혼입점검등
입고QA
현대물류센터입고상품대상, 샘플링점검을통한입고가능여부최종검사
기준일
의뢰: 방송일기준최소3일전의뢰
업무
구성품혼입, 상품오염, 내외관불량표시, 정보오류점검등
○QA관련파트너시스템주요화면
QA 자료관리(VD036)
QA 의뢰및승인현황을확인하는화면입니다.
①의뢰상품의QA관리코드가등록되면,
품질서류파일을업로드할수있습니다.
②파일별진행상태확인이가

7. QA
○단계별QA 종류
구분
내용
사전QA
상품별서류및샘플점검을통한상품전반의품질관리업무를총괄
기준일
의뢰: 방송일기준최소14일전의뢰/ 완료: 방송일기준최소4일전완료
업무
법적준수사항이행여부확인, 품질서류검토,
원산지검증, 방송품질소구관련증빙자료검토등
현장QA
제조사, 물류센터방문을통한완제품의진정성을확인
기준일
의뢰: 방송일기준최소10일전의뢰/ 완료: 방송일기준최소4일전완료
업무
제조사위생점검및원료및자재관리점검, 품질표시정보확인,
완제품규격및포장상태점검, 외관불량및구성품혼입점검등
입고QA
현대물류센터입고상품대상, 샘플링점검을통한입고가능여부최종검사
기준일
의뢰: 방송일기준최소3일전의뢰
업무
구성품혼입, 상품오염, 내외관불량표시, 정보오류점검등
○QA관련파트너시스템주요화면
QA 자료관리(VD036)
QA 의뢰및승인현황을확인하는화면입니다.
①의뢰상품의QA관리코드가등록되면,
품질서류파일을업로드할수있습니다.
②파일별진행상태확인이가능합니다.
(미등록/ 등록/ 수정요청/ 승인완료)
③‘수정요청’ 시파일재업로드가필요합니다.


|구분|내용|Col3|
|---|---|---|
|**사전QA**|**상품별서류및샘플점검을통한상품전반의품질관리업무를총괄**|**상품별서류및샘플점검을통한상품전반의품질관리업무를총괄**|
|**사전QA**|**기준일**|**의뢰: 방송일기준최소14일전의뢰/ 완료: 방송일기준최소4일전완료**|
|**사전QA**|**업무**|**법적준수사항이행여부확인, 품질서류검토,**<br>**원산지검증, 방송품질소구관련증빙자료검토등**|
|**현장QA**|**제조사, 물류센터방문을통한완제품의진정성을확인**|**제조사, 물류센터방문을통한완제품의진정성을확인**|
|**현장QA**|**기준일**|**의뢰: 방송일기준최소10일전의뢰/ 완료: 방송일기준최소4일전완료**|
|**현장QA**|**업무**|**제조사위생점검및원료및자재관리점검, 품질표시정보확인,**<br>**완제품규격및포장상태점검, 외관불량및구성품혼입점검등**|
|**입고QA**|**현대물류센터입고상품대상, 샘플링점검을통한입고가능여부최종검사**|**현대물류센터입고상품대상, 샘플링점검을통한입고가능여부최종검사**|
|**입고QA**|**기준일**|**의뢰: 방송일기준최소3일전의뢰**|
|**입고QA**|**업무**|**구성품혼입, 상품오염, 내외관불량표시, 정보오류점검등**|

----------------------------------------------------------------------------------------------------


In [11]:
extract_all_markdown_tables(PDF_LIST[0])

|방송<br>LIVE / DATA|· 100여개의방송채널을통해전국으로송출<br>· LIVE: 실시간영상/ DATA: 녹화영상<br>· 경쟁력있는고정PGM 운영<br>· 생활, 패션, 트렌드, 렌탈, 금융등<br>· 주문방식: 상담원, ARS, WEB, 모바일|
|---|---|


|Hmall<br>온라인몰|· 온라인전용판매매체<br>(방송상품동시판매가능)<br>· 다양한상품군을<br>구성제한없이판매<br>· 광고/마케팅/제휴지원<br>· 주문방식: WEB, 모바일|
|---|---|


|쇼라<br>모바일라이브|· 쌍방향소통이가능한<br>모바일전용라이브커머스<br>· MZ 고객층과의가교역할<br>· 대표브랜딩‘믿고사쇼라’<br>· 주문방식: 모바일|
|---|---|

|단계|내용|
|---|---|
|**협력사제안**|**협력사가파트너시스템을통해신상품제안**|
|**상품평가및입점**|**MD가상품검토후, 최종입점결정**|
|**상품등록**|**상품정보입력**|
|**QA**|**현장및사전QA**|
|**방송편성**|**상품군, 목표등을고려하여방송시간편성**|
|**심의**|**영상및자막에대한심의검토**|
|**사전전략회의**|**상품특장점및소구포인트등방송진행방향논의**|
|**방송판매**|**Live 방송진행**|
|**사후전략회의**|**방송실적분석, 개선점공유및차후운영방향협의**|
|**배송**|**출고및배송체크**|
|**CS**|**고객불만접수및처리**|
|**정산**|**대금지급**|


|단계|내용|
|---|---|
|**협력사제안**|**협력사가파트너시스템을통해신상품제안**|
|**상품평가및입점**|**MD가상품검토후, 최종입점결정**|
|**상품등록**|**상품정보입력**|
|**QA/심의**|**웹기술서심의**|
|**(프로모션)**|**상품및행사에맞는쿠폰/적립/무이자등판촉협의**|
|**(광고)**|**상품및행사에맞는배너/탭/팝업등광고협의**|
|**(링크/DB 제휴)**|**가격비교사이트및타쇼핑몰노출협의**|
|**판매**|**Hmall노출**|
|**사후관리**|**실적분석, 행사/기획전리뷰**|
|**배송**|**출고및배송체크**|
|**CS**|**고객불만접수및처리**|
|**정산**|**대금지급**|

|Col1|Col2|Col3|
|---|---|---|
||||
||||

|Col1|Col2|
|---|---|
|**필독으**||
|||

|① QA 의뢰|② QA 자료 업로드|③ QA 검토|
|---|---|---|


|⑥ 사후 관리|⑦ 검사|
|---|---|

...

|화면명(번호)|Col2|내용|
|---|---|---|
|**협력사Ref접수**|**(VD401)**|**협력사에서당사로확인요청**|
|**협력사Ref처리**|**(VD402)**|**당사에서요청된건에대한답변입력**|
|**택배사Ref접수**|**(VD923)**|**택배사에서당사로확인요청**|
|**택배사Ref처리**|**(VD924)**|**당사에서요청된건에대한답변입력**|
|**AS처리**|**(VD404)**|**당사에서요청된건에대한답변입력**|
|**드림CS**|**(VD403)**|**소비자의의견을경영자(임원)가직접처리및내부공유**|
|**품질이상처리**|**(VD913)**|**품질관련불만이판매량대비2% 이상접수되는상품확인**|
|**불만접수현황(사유별)**|**(VD703)**|**협력사에서진행하는상품들의불만현황파악**|
|**Ref미처리현황(실시간)**|**(VD704)**|**실시간미처리현황**|

|구분|내용|
|---|---|
|**사업자**<br>**등록증**|**발행일자구분없음**<br>**※ 개인사업자의경우주민등록등본(원본), 신분증사본**|
|**통장**<br>**사본**|**당사협력사현황표에기재된사업자명표시필수(★인감날인본)**<br>**※ 개인사업자의경우대표자명과회사명함께기재가능**|
|**이행(지급)**<br>**보증보험**|**· 일반상품군: 방송/데이터방송1천만원~, 인터넷5백만원~**<br>**· A/S상품군: 방송/데이터방송5천만원~, 인터넷1천만원~**|
|**이행(지급)**<br>**보증보험**|**최초2년가입1년단위추가갱신/ 1년이상보증보험기간상시유지**|


|구분|내용|
|---|---|
|**정산**<br>**주기**|**· 1주기: 01일~ 10일**<br>**· 2주기: 11일~ 20일**<br>**· 3주기: 21일~ 31일(말일)**|
|**대금**<br>**정산**|**· 공급가정산:주기별마감후영업일기준+5일**<br>**※ 대기업의경우정산주기마감+25 영업일뒤지급**<br>**· 부가세정산: 익월10일세금계산서승인마감후+5영업일**<br>**※ 대기업의경우매월1주기대금지급일**|
|**정산**<br>**기준**|**· 특약: 주문출고확정일기준매출발생**<br>**· 직매입: 입고확정일기준매출발생**|

|구분|내용|Col3|Col4|
|---|---|---|---|
|**발행방식**|**정발행**|**상품을공급하는자→ 상품을공급받는자**|**상품을공급하는자→ 상품을공급받는자**|
|**발행방식**|**역발행**|**상품을공급받는자(당사)가세금계산서발행**<br>**상품을공급하는자(협력사)가세금계산서승인**|**상품을공급받는자(당사)가세금계산서발행**<br>**상품을공급하는자(협력사)가세금계산서승인**|
|**마감**|**당사**|**매월1일마감세금계산서발행**|**매월1일마감세금계산서발행**|
|**마감**|**협력사**|**‘매월1~10일’세금계산서인증필요**<br>**※ 미인증시상품대금부가세지급불가(세무신고불가)**|**‘매월1~10일’세금계산서인증필요**<br>**※ 미인증시상품대금부가세지급불가(세무신고불가)**|
|**부가세**<br>**지급**|**부가세금액은월1회(매월10일)인증값을기준으로마감,**<br>**익월중순경해당월1주기상품대금과합산하여지급**|**부가세금액은월1회(매월10일)인증값을기준으로마감,**<br>**익월중순경해당월1주기상품대금과합산하여지급**|**부가세금액은월1회(매월10일)인증값을기준으로마감,**<br>**익월중순경해당월1주기상품대금과합산하여지급**|
|**발행금액**<br>**(배송형태별)**|**물류센터입고**|**물류센터입고**|**물류센터입고분(당사매입분)**|
|**발행금액**<br>**(배송형태별)**|**직접배송**|**직접배송**|**출고분(상품대금지급대상금액)**|


|화면명(번호)|내용|
|---|---|
|**매입매출현황**<br>**(VD603)**|**협력사의기간별매입/매출내역파악**|
|**대금지불내역**<br>**(VD606)**|**협력사의주기별정산내역확인**|
|**공제상세현황**<br>**(VD607)**|**협력사의주기별공제항목별상세내역확인**|
|**전자세금계산서**<br>**(DP502)**|**세금계산서종류별발행내역확인및**<br>**세금계산서승인화면접속**|
|**전자채권채무조회**<br>**(AF341)**|**협력사의채권채무조회서전자발급신청**|

|Hmall제휴|홍보 / 브랜드마케팅|Col3|
|---|---|---|
|**○신규매체/플랫폼전략제휴**<br>**○판매수수료기반CPS(Cost per sale) 제휴**|**○언론홍보/보도자료**<br>**○온라인바이럴/디지털광고**|**○방송행사및대외이벤트**<br>**○친환경캠페인**|
|**▷담당자연락처yunsh_@hyundaihmall.com**|**▷담당자연락처jhdo919@hyundaihmall.com**|**▷담당자연락처jhdo919@hyundaihmall.com**|
|**퍼포먼스마케팅**|**Hmall내부광고**|**Hmall내부광고**|
|**○방송/Hmall신규광고및플랫폼제안**<br>**○검색광고-키워드/쇼핑검색등**<br>**○퍼포먼스광고-DA/APP/SNS**|**○HmallPC/모바일광고**<br>**○Hmall배너광고**<br>**광고상품소개서바로가기**|**○HmallPC/모바일광고**<br>**○Hmall배너광고**<br>**광고상품소개서바로가기**|
|**▷담당자연락처chasy@hyundaihmall.com**|**▷담당자연락처kmsong1523377@hyundaihmall.com**|**▷담당자연락처kmsong1523377@hyundaihmall.com**|

[{'page': 3,
  'table_no': 1,
  'markdown': '|방송<br>LIVE / DATA|· 100여개의방송채널을통해전국으로송출<br>· LIVE: 실시간영상/ DATA: 녹화영상<br>· 경쟁력있는고정PGM 운영<br>· 생활, 패션, 트렌드, 렌탈, 금융등<br>· 주문방식: 상담원, ARS, WEB, 모바일|\n|---|---|\n\n\n|Hmall<br>온라인몰|· 온라인전용판매매체<br>(방송상품동시판매가능)<br>· 다양한상품군을<br>구성제한없이판매<br>· 광고/마케팅/제휴지원<br>· 주문방식: WEB, 모바일|\n|---|---|\n\n\n|쇼라<br>모바일라이브|· 쌍방향소통이가능한<br>모바일전용라이브커머스<br>· MZ 고객층과의가교역할<br>· 대표브랜딩‘믿고사쇼라’<br>· 주문방식: 모바일|\n|---|---|'},
 {'page': 4,
  'table_no': 1,
  'markdown': '|단계|내용|\n|---|---|\n|**협력사제안**|**협력사가파트너시스템을통해신상품제안**|\n|**상품평가및입점**|**MD가상품검토후, 최종입점결정**|\n|**상품등록**|**상품정보입력**|\n|**QA**|**현장및사전QA**|\n|**방송편성**|**상품군, 목표등을고려하여방송시간편성**|\n|**심의**|**영상및자막에대한심의검토**|\n|**사전전략회의**|**상품특장점및소구포인트등방송진행방향논의**|\n|**방송판매**|**Live 방송진행**|\n|**사후전략회의**|**방송실적분석, 개선점공유및차후운영방향협의**|\n|**배송**|**출고및배송체크**|\n|**CS**|**고객불만접수및처리**|\n|**정산**|**대금지급**|\n\n\n|단계|내용|\n|---|---|\n|**협력사제안**|**협력사가파트너시스템을통해신상품제안**|\n|**상품평가및입점**|**MD가상품검토후, 최종입점결정**|\n|**상품등록**|**상품정보입력**|\n|**QA/심의**|**웹기술서심의**|\n|**(프

In [ ]:
#2. hmall 소개서 : 표 헤더밖에 못가져옴_이미지로 처리 필요 / 3~26 => PDFPlumberLoader  (위치 그대로 가져옴)
show_page_content(pdf_path=PDF_LIST[1], page_num=27)

In [ ]:
extract_all_markdown_tables(PDF_LIST[1])

In [12]:
#3. 쇼라소개서 : 아무것도 못읽음 -> 일단 제외
show_page_content(pdf_path=PDF_LIST[2], page_num=3)

PyPDFLoader - 3페이지

문자수: 0
metadata: {'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-04-03T16:14:08+09:00', 'title': 'PowerPoint 프레젠테이션', 'author': '하유림 선임', 'moddate': '2024-04-03T16:14:08+09:00', 'fasoo_trace_id': 'eyAibm9kZUNvdW50IjogOSwgIm5vZGUxIiA6IHsiZHNkIjoiMDEwMDAwMDAwMDAwMTg5NCIsImxvZ1RpbWUiOiIyMDI0LTA0LTAzVDAwOjQxOjQxWiIsInBJRCI6MSwidHJhY2VJZCI6IjFEOUNDQUQwRjlDMzREQzJCNDk1NEZBNzNGOTJGQTVFIiwidXNlckNvZGUiOiIyMjExMzcxIn0sIm5vZGUyIiA6IHsiZHNkIjoiMDEwMDAwMDAwMDAwMTg5NCIsImxvZ1RpbWUiOiIyMDI0LTA0LTAzVDAxOjE1OjI1WiIsInBJRCI6MSwidHJhY2VJZCI6IjY5RUEyRjgyN0FEODRGMTNCNTA2QjRCRTAxMEVFMjZFIiwidXNlckNvZGUiOiIyMjExMzcxIn0sIm5vZGUzIiA6IHsiZHNkIjoiMDEwMDAwMDAwMDAwMTg5NCIsImxvZ1RpbWUiOiIyMDI0LTA0LTAzVDA3OjAyOjUzWiIsInBJRCI6MSwidHJhY2VJZCI6IjM1Qzk1MTBEMzBERDQzQTI5OThEOThFN0MwMzE4RjM3IiwidXNlckNvZGUiOiIyMjExMzcxIn0sIm5vZGU0IiA6IHsiZHNkIjoiMDEwMDAwMDAwMDAwMTg5NCIsImxvZ1RpbWUiOiIyMDI0LTA0LTAzVDA3OjA2OjE3WiIsInBJRCI6MSwidHJhY2VJZCI6IjM0MEE3MTA3RTY4RDRCNEI4RjQ

----------------------------------------------------------------------------------------------------


PDFPlumberLoader - 3페이지

문자수: 1
metadata: {'source': 'manual/3. 쇼라 소개서 (p.22).pdf', 'file_path': 'manual/3. 쇼라 소개서 (p.22).pdf', 'page': 2, 'total_pages': 22, 'Title': 'PowerPoint 프레젠테이션', 'Author': '하유림 선임', 'CreationDate': "D:20240403161408+09'00'", 'ModDate': "D:20240403161408+09'00'", 'Producer': 'Microsoft® PowerPoint® 2016', 'Creator': 'Microsoft® PowerPoint® 2016', 'Fasoo_Trace_ID': 'eyAibm9kZUNvdW50IjogOSwgIm5vZGUxIiA6IHsiZHNkIjoiMDEwMDAwMDAwMDAwMTg5NCIsImxvZ1RpbWUiOiIyMDI0LTA0LTAzVDAwOjQxOjQxWiIsInBJRCI6MSwidHJhY2VJZCI6IjFEOUNDQUQwRjlDMzREQzJCNDk1NEZBNzNGOTJGQTVFIiwidXNlckNvZGUiOiIyMjExMzcxIn0sIm5vZGUyIiA6IHsiZHNkIjoiMDEwMDAwMDAwMDAwMTg5NCIsImxvZ1RpbWUiOiIyMDI0LTA0LTAzVDAxOjE1OjI1WiIsInBJRCI6MSwidHJhY2VJZCI6IjY5RUEyRjgyN0FEODRGMTNCNTA2QjRCRTAxMEVFMjZFIiwidXNlckNvZGUiOiIyMjExMzcxIn0sIm5vZGUzIiA6IHsiZHNkIjoiMDEwMDAwMDAwMDAwMTg5NCIsImxvZ1RpbWUiOiIyMDI0LTA0LTAzVDA3OjAyOjUzWiIsInBJRCI6MSwidHJhY2VJZCI6IjM1Qzk1MTBEMzBERDQzQTI5OThEOThFN0MwMzE4RjM3IiwidXNlckNvZGUiOiIyMjExMzcxIn0sIm5vZGU0IiA6IHsiZHNkIjoiMDEwMDAw

...

문자수: 0
metadata: {'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-04-03T16:14:08+09:00', 'source': 'manual/3. 쇼라 소개서 (p.22).pdf', 'file_path': 'manual/3. 쇼라 소개서 (p.22).pdf', 'total_pages': 22, 'format': 'PDF 1.5', 'title': 'PowerPoint 프레젠테이션', 'author': '하유림 선임', 'subject': '', 'keywords': '', 'moddate': '2024-04-03T16:14:08+09:00', 'trapped': '', 'modDate': "D:20240403161408+09'00'", 'creationDate': "D:20240403161408+09'00'", 'page': 2}
page_content: 


----------------------------------------------------------------------------------------------------


PyMuPDF + Markdown - 3페이지

문자수: 0
metadata: {'producer': 'Microsoft® PowerPoint® 2016', 'creator': 'Microsoft® PowerPoint® 2016', 'creationdate': '2024-04-03T16:14:08+09:00', 'source': 'manual/3. 쇼라 소개서 (p.22).pdf', 'file_path': 'manual/3. 쇼라 소개서 (p.22).pdf', 'total_pages': 22, 'format': 'PDF 1.5', 'title': 'PowerPoint 프레젠테이션', 'author': '하유림 선임', 'subject': '', 'keywords': '', 'moddate': '2024-04-03T16:14:08+09:00', 'trapped': '', 'modDate': "D:20240403161408+09'00'", 'creationDate': "D:20240403161408+09'00'", 'page': 2}
page_content: 


----------------------------------------------------------------------------------------------------


In [13]:
#4. 광고상품소개서 : 표 생성이 제대로 안됨 / 2~23 -> PDFPlumberLoader 
show_page_content(pdf_path=PDF_LIST[3], page_num=5)

PyPDFLoader - 5페이지

문자수: 429
metadata: {'producer': 'Microsoft® PowerPoint® Microsoft 365용', 'creator': 'Microsoft® PowerPoint® Microsoft 365용', 'creationdate': '2026-08-27T16:53:20+09:00', 'title': 'PowerPoint 프레젠테이션', 'author': 'Hhome', 'moddate': '2026-08-27T16:53:20+09:00', 'source': 'manual/4. 광고 상품 소개서 (p.24).pdf', 'total_pages': 24, 'page': 4, 'page_label': '5'}
page_content: D_월간신상
(추천브랜드)
C_타임딜
영역별 광고 단가 – 현대홈쇼핑탭
노출도가 높은 영역으로 브랜드 마케팅 메시지를 효과적으로 전달할 수 있습니다.
4
구분 상품명 단가 (만원) 노출기간 구좌수(日)
현대
홈쇼핑탭
A_탑배너 (상단) 300만원 1일 2개
A_탑배너 (중단) 150만원 1일 3개
A_탑배너 (하단) 100만원 1일 4개
B_오늘의 메인 행사 180만원 1일 1개
C_타임딜 150만원 1일 1개
D_월간 신상 70만원 1일 3개
E_MD 제안 20만원 1일 2개
F_오늘의 추천 상품 30만원 1일 1개
G_지금 주목할 기획전 (상단) 120만원 1일 1개
G_지금 주목할 기획전 (중단) 100만원 1일 2개
H_믿고 보는 혜택상품 (상단) 100만원 1일 2개
H_믿고 보는 혜택상품 (중단) 50만원 1일 4개
I_오늘특가 2만원 1일 무제한


D_월간신상
(추천브랜드)
C_타임딜
영역별 광고 단가 – 현대홈쇼핑탭
노출도가 높은 영역으로 브랜드 마케팅 메시지를 효과적으로 전달할 수 있습니다.
4
구분 상품명 단가 (만원) 노출기간 구좌수(日)
현대
홈쇼핑탭
A_탑배너 (상단) 300만원 1일 2개
A_탑배너 (중단) 150만원 1일 3개
A_탑배너 (하단) 100만원 1일 4개
B_오늘의 메인 행사 180만원 1일 1개
C_타임딜 150만원 1일 1개
D_월간 신상 70만원 1일 3개
E_MD 제안 20만원 1일 2개
F_오늘의 추천 상품 30만원 1일 1개
G_지금 주목할 기획전 (상단) 120만원 1일 1개
G_지금 주목할 기획전 (중단) 100만원 1일 2개
H_믿고 보는 혜택상품 (상단) 100만원 1일 2개
H_믿고 보는 혜택상품 (중단) 50만원 1일 4개
I_오늘특가 2만원 1일 무제한

----------------------------------------------------------------------------------------------------


PDFPlumberLoader - 5페이지

...

----------------------------------------------------------------------------------------------------


PyMuPDF + Markdown - 5페이지

문자수: 2576
metadata: {'producer': 'Microsoft® PowerPoint® Microsoft 365용', 'creator': 'Microsoft® PowerPoint® Microsoft 365용', 'creationdate': '2026-08-27T16:53:20+09:00', 'source': 'manual/4. 광고 상품 소개서 (p.24).pdf', 'file_path': 'manual/4. 광고 상품 소개서 (p.24).pdf', 'total_pages': 24, 'format': 'PDF 1.7', 'title': 'PowerPoint 프레젠테이션', 'author': 'Hhome', 'subject': '', 'keywords': '', 'moddate': '2026-08-27T16:53:20+09:00', 'trapped': '', 'modDate': "D:20260827165320+09'00'", 'creationDate': "D:20260827165320+09'00'", 'page': 4}
page_content: D_월간신상
(추천브랜드)
C_타임딜
영역별광고단가– 현대홈쇼핑탭
노출도가높은영역으로브랜드마케팅메시지를효과적으로전달할수있습니다.
4
구분
상품명
단가(만원) 노출기간구좌수(日)
현대
홈쇼핑탭
A_탑배너(상단)
300만원
1일
2개
A_탑배너(중단)
150만원
1일
3개
A_탑배너(하단)
100만원
1일
4개
B_오늘의메인행사
180만원
1일
1개
C_타임딜
150만원
1일
1개
D_월간신상
70만원
1일
3개
E_MD 제안
20만원
1일
2개
F_오늘의추천상품
30만원
1일
1개
G_지금주목할기획전(상단)
120만원
1일
1개
G_지금주목할기획전(중단)
100만원
1일
2개
H_믿고보는혜택상품(상단)
100만원
1일
2개
H_믿고보는혜택상품(중단)
50만원
1일
4개
I_오늘특가
2만원
1일
무제한


|구분<br>A 탑배|상품명 단가<br>너(상단) 3|(만원) 노출기간<br>00만원 1일|
|---|--

D_월간신상
(추천브랜드)
C_타임딜
영역별광고단가– 현대홈쇼핑탭
노출도가높은영역으로브랜드마케팅메시지를효과적으로전달할수있습니다.
4
구분
상품명
단가(만원) 노출기간구좌수(日)
현대
홈쇼핑탭
A_탑배너(상단)
300만원
1일
2개
A_탑배너(중단)
150만원
1일
3개
A_탑배너(하단)
100만원
1일
4개
B_오늘의메인행사
180만원
1일
1개
C_타임딜
150만원
1일
1개
D_월간신상
70만원
1일
3개
E_MD 제안
20만원
1일
2개
F_오늘의추천상품
30만원
1일
1개
G_지금주목할기획전(상단)
120만원
1일
1개
G_지금주목할기획전(중단)
100만원
1일
2개
H_믿고보는혜택상품(상단)
100만원
1일
2개
H_믿고보는혜택상품(중단)
50만원
1일
4개
I_오늘특가
2만원
1일
무제한


|구분<br>A 탑배|상품명 단가<br>너(상단) 3|(만원) 노출기간<br>00만원 1일|
|---|---|---|
|~~**현**~~**대**<br>~~**홈**~~**쇼핑**~~**탭**~~<br>_<br>A_~~탑~~배<br>A_~~탑~~배<br>B_오~~늘~~<br>C_타~~임~~<br>D_월간<br>E_MD<br>F_오~~늘~~<br>G_지~~금~~<br>G_지~~금~~<br>H_믿고<br>H_믿고<br>I_오~~늘특~~|너(중단)<br>1|50만원<br>1일|
|~~**현**~~**대**<br>~~**홈**~~**쇼핑**~~**탭**~~<br>_<br>A_~~탑~~배<br>A_~~탑~~배<br>B_오~~늘~~<br>C_타~~임~~<br>D_월간<br>E_MD<br>F_오~~늘~~<br>G_지~~금~~<br>G_지~~금~~<br>H_믿고<br>H_믿고<br>I_오~~늘특~~|너(~~하~~단)<br>1|00만원<br>1일|
|~~**현**~~**대**<br>~~**홈**~~**쇼핑**~~**탭**~~<br>_<br>A_~~탑~~배<br>A_~~탑~~배<br>B_오~~늘~~<br>C_타~~임~~<br>D_월간<br>E_MD<br>F_오~~늘~~<br>G_지~~금~~<br>G_지~~금~~<br>H_믿고<br>H_믿고<br>I_오~~늘특~~|의메인~~행~~사<br>1|80만원<br>1일|
|~~**현**~~**대**<br>~~**홈**~~**쇼핑**~~**탭**~~<br>_<br>A_~~탑~~배<br>A_~~탑~~배<br>B_오~~늘~~<br>C_타~~임~~<br>D_월간<br>E_MD<br>F_오~~늘~~<br>G_지~~금~~<br>G_지~~금~~<br>H_믿고<br>H_믿고<br>I_오~~늘특~~|딜<br>1|50만원<br>1일|
|~~**현**~~**대**<br>~~**홈**~~**쇼핑**~~**탭**~~<br>_<br>A_~~탑~~배<br>A_~~탑~~배<br>B_오~~늘~~<br>C_타~~임~~<br>D_월간<br>E_MD<br>F_오~~늘~~<br>G_지~~금~~<br>G_지~~금~~<br>H_믿고<br>H_믿고<br>I_오~~늘특~~|신상<br><br> 제안<br>|70만원<br>1일<br>0만원<br>1일|
|~~**현**~~**대**<br>~~**홈**~~**쇼핑**~~**탭**~~<br>_<br>A_~~탑~~배<br>A_~~탑~~배<br>B_오~~늘~~<br>C_타~~임~~<br>D_월간<br>E_MD<br>F_오~~늘~~<br>G_지~~금~~<br>G_지~~금~~<br>H_믿고<br>H_믿고<br>I_오~~늘특~~|<br><br>의추천상~~품~~<br>|30만원<br>1일|
|~~**현**~~**대**<br>~~**홈**~~**쇼핑**~~**탭**~~<br>_<br>A_~~탑~~배<br>A_~~탑~~배<br>B_오~~늘~~<br>C_타~~임~~<br>D_월간<br>E_MD<br>F_오~~늘~~<br>G_지~~금~~<br>G_지~~금~~<br>H_믿고<br>H_믿고<br>I_오~~늘특~~|주~~목할~~기~~획~~전(상단)<br>1|20만원<br>1일|
|~~**현**~~**대**<br>~~**홈**~~**쇼핑**~~**탭**~~<br>_<br>A_~~탑~~배<br>A_~~탑~~배<br>B_오~~늘~~<br>C_타~~임~~<br>D_월간<br>E_MD<br>F_오~~늘~~<br>G_지~~금~~<br>G_지~~금~~<br>H_믿고<br>H_믿고<br>I_오~~늘특~~|주~~목할~~기~~획~~전(중단)<br>1<br>~~보는혜~~택상~~품~~(상단)<br>1|00만원<br>1일<br>0만원<br>1일|
|~~**현**~~**대**<br>~~**홈**~~**쇼핑**~~**탭**~~<br>_<br>A_~~탑~~배<br>A_~~탑~~배<br>B_오~~늘~~<br>C_타~~임~~<br>D_월간<br>E_MD<br>F_오~~늘~~<br>G_지~~금~~<br>G_지~~금~~<br>H_믿고<br>H_믿고<br>I_오~~늘특~~|~~보는혜~~택상~~품~~(중단)<br>|50만원<br>1일|
|~~**현**~~**대**<br>~~**홈**~~**쇼핑**~~**탭**~~<br>_<br>A_~~탑~~배<br>A_~~탑~~배<br>B_오~~늘~~<br>C_타~~임~~<br>D_월간<br>E_MD<br>F_오~~늘~~<br>G_지~~금~~<br>G_지~~금~~<br>H_믿고<br>H_믿고<br>I_오~~늘특~~|가|2만원<br>1일|

----------------------------------------------------------------------------------------------------


In [ ]:
extract_all_markdown_tables(PDF_LIST[3])

In [ ]:
#5. 협력사시스템 사용설명서 : 아무것도 못읽음 -> 일단 제외
show_page_content(pdf_path=PDF_LIST[4], page_num=3)

In [ ]:
#6. 신규협력사 입점절차 : PyMu + Markdown / 2~21
show_page_content(pdf_path=PDF_LIST[5], page_num=17)

In [ ]:
extract_all_markdown_tables(PDF_LIST[5])

In [ ]:
#7. 데이터영역 광고제안서 : PyMu + Markdown / 2~7
show_page_content(pdf_path=PDF_LIST[6], page_num=6)

In [ ]:
extract_all_markdown_tables(PDF_LIST[6])

In [ ]:
#8. QA 가이드 : PyMu + Markdown / 4~8, 10~15, 17~18, 20~27, 29~34, 36~37
show_page_content(pdf_path=PDF_LIST[7], page_num=37)

In [ ]:
extract_all_markdown_tables(PDF_LIST[7])

In [31]:
import pdfplumber

with pdfplumber.open(PDF_LIST[3]) as pdf:
    tables = pdf.pages[4].extract_tables()
print(len(tables), "개 표")

1 개 표


In [34]:
for row in tables[0][:5]:
    print(" ", [(cell or "")[:20].replace("\n", " ") for cell in row])

  ['구분', '상품명 A_탑배너(상단)', '단가(만원) 노출기간 구 300만원 ', '']
  ['현대 홈쇼핑', 'A_탑배너(중단) A_탑배너(하단) ', '150만원 1일', '']
  ['', '', '100만원 1일', '']
  ['', '', '180만원 1일', '']
  ['', '', '150만원 1일', '']


표 추출 함께 사용하는 경우
page.find_tables()
→ table.bbox
→ 표 영역 text 제외
→ table.extract()

## Chunking

In [53]:
PDF_CONFIGS = {
    "manual/1. 협력사 운영 통합 안내서 (p.23).pdf": "pymupdf_markdown",
    "manual/2. Hmall 소개서 (p.30).pdf": "pdfplumber",
    "manual/4. 광고 상품 소개서 (p.24).pdf": "pdfplumber",
    "manual/6. 신규 협력사 입점 절차 안내 (p.21).pdf": "pymupdf_markdown",
    "manual/7. 데이터 영역 광고 제안서 (p.8).pdf": "pymupdf_markdown",
    "manual/8. QA가이드 (p.21).pdf": "pymupdf_markdown",
}

***TODO : 목차 빼야함

In [62]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def load_pymupdf_markdown(pdf_path) :
    markdown_PyMuPDFLoader = PyMuPDFLoader(
                                pdf_path,
                                mode="page",
                                extract_tables="markdown"
                            )
    docs = markdown_PyMuPDFLoader.load()
    return docs

def load_pdfplumber(pdf_path) :
    docs = PDFPlumberLoader(pdf_path).load()
    return docs

all_chunks = []
for pdf_idx, pdf_path in enumerate(PDF_LIST):
    pdf_path = str(pdf_path)
    if pdf_path not in PDF_CONFIGS:
        continue

    method = PDF_CONFIGS[pdf_path]
    print(pdf_idx, method)

    # 1. PDF별 로딩
    print(pdf_path)
    if method == "pymupdf_markdown":
        docs = load_pymupdf_markdown(pdf_path)

    elif method == "pdfplumber":
        docs = load_pdfplumber(pdf_path)

    # 2. PDF 특성에 맞게 청킹
    chunks = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=80
    ).split_documents(docs)
    print(len(chunks))

    # 3. metadata 추가
    for chunk in chunks:
        chunk.metadata["source"] = pdf_path.split("manual/", 1)[1]
        chunk.metadata["loader"] = method

    all_chunks.extend(chunks)

0 pymupdf_markdown
manual/1. 협력사 운영 통합 안내서 (p.23).pdf
47
1 pdfplumber
manual/2. Hmall 소개서 (p.30).pdf
30
3 pdfplumber
manual/4. 광고 상품 소개서 (p.24).pdf
28
5 pymupdf_markdown
manual/6. 신규 협력사 입점 절차 안내 (p.21).pdf
25
6 pymupdf_markdown
manual/7. 데이터 영역 광고 제안서 (p.8).pdf
10
7 pymupdf_markdown
manual/8. QA가이드 (p.21).pdf
135


In [ ]:
all_chunks

## Embedding & Vector store

In [80]:
# 벡터스토어별 비교 : 인메모리, 크로마, FAISS, Qdrant

from langchain_chroma import Chroma
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.vectorstores import FAISS
from langchain_qdrant import QdrantVectorStore

#chroma.delete_collection()

memory = InMemoryVectorStore.from_documents(all_chunks, embeddings)
chroma = Chroma.from_documents(all_chunks, embeddings, collection_name="hyundaihomeshopping_vendor_manual_chroma")
faiss_store = FAISS.from_documents(all_chunks, embeddings)
qdrant = QdrantVectorStore.from_documents(all_chunks, embeddings, location=":memory:", collection_name="hyundaihomeshopping_vendor_manual_qdrant")
Q = "현대홈쇼핑에 신규 입점하려면 어떻게 해?"

In [88]:
for name, store in [("InMemory", memory), ("Chroma", chroma), ("FAISS", faiss_store), ("Qdrant", qdrant)]:
    hits = store.similarity_search_with_score(Q, k=3)
    print(f"{name:9}", [(d.metadata["source"], d.metadata["page"], round(float(s), 3)) for d, s in hits])

InMemory  [('6. 신규 협력사 입점 절차 안내 (p.21).pdf', 0, 0.655), ('6. 신규 협력사 입점 절차 안내 (p.21).pdf', 1, 0.539), ('1. 협력사 운영 통합 안내서 (p.23).pdf', 9, 0.537)]
Chroma    [('6. 신규 협력사 입점 절차 안내 (p.21).pdf', 0, 0.691), ('6. 신규 협력사 입점 절차 안내 (p.21).pdf', 0, 0.691), ('6. 신규 협력사 입점 절차 안내 (p.21).pdf', 0, 0.691)]
FAISS     [('6. 신규 협력사 입점 절차 안내 (p.21).pdf', 0, 0.691), ('6. 신규 협력사 입점 절차 안내 (p.21).pdf', 1, 0.923), ('1. 협력사 운영 통합 안내서 (p.23).pdf', 9, 0.926)]
Qdrant    [('6. 신규 협력사 입점 절차 안내 (p.21).pdf', 0, 0.655), ('6. 신규 협력사 입점 절차 안내 (p.21).pdf', 1, 0.539), ('1. 협력사 운영 통합 안내서 (p.23).pdf', 9, 0.537)]


## Naive RAG

In [84]:
retriever = faiss_store.as_retriever(search_kwargs={"k":3})
context = retriever.invoke(Q)
for doc in context:
    print(doc.metadata["page"], doc.page_content[:100].replace("\n", " "))

0 신규협력사 입점절차안내 현대홈쇼핑
1 가입여부확인및기초가입하기 현대홈쇼핑입점프로세스 신규협력사의신상품제안프로세스는, 협력사시스템의[ 신규입점제안] 메뉴를통해이루어집니다. !
9 5. 신규입점 ○신규입점은파트너시스템에서진행됩니다.신규입점매뉴얼을참고해주세요. Hmall홈페이지최하단[신규입점] 탭에서도절차확인이가능합니다. 입점ID/PW 확인 전자계약서등록및작성


In [86]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

answer_prompt = ChatPromptTemplate.from_messages([
    ("system", "근거만으로 답한다. 근거에 없으면 '문서에 없음' 이라고 답한다."),
    ("human", "근거\n{context}\n\n질문\n{question}"),
])
answers = answer_prompt | ChatOpenAI(model=MODEL) | StrOutputParser()
answer = answers.invoke({
    "context": context,
    "question": Q
})
print(answer)

현대홈쇼핑 신규 입점은 **파트너시스템/협력사시스템의 [신규입점제안] 메뉴**를 통해 진행됩니다.

문서 근거상 절차는 다음과 같습니다.

1. **파트너시스템 접속**
2. **[신규입점제안] 클릭**
3. **협력사 정보 입력**
4. **상품 제안 및 결과 확인**
5. **전자계약서 등록 및 작성**

추가로 문서에는 아래 내용도 있습니다.
- **신규입점은 파트너시스템에서 진행**
- **신규입점 매뉴얼 참고**
- **Hmall 홈페이지 최하단 [신규입점] 탭에서도 절차 확인 가능**

질문하신 “어떻게 해?”에 대한 요약:
- **협력사시스템/파트너시스템에 접속해서 [신규입점제안] 메뉴로 입점 절차를 시작하면 됩니다.**


## golden dataset

In [100]:
TARGET = {"단일홉": 6, "멀티홉": 4, "집계": 3, "조건" : 2, "절차" : 2,  "무답변": 3}
OVERSAMPLE = 2  # 검수에서 절반 가까이 떨어진다. 처음부터 넉넉히 만든다 (2배 정도로 미리 만든다)
print(sum(TARGET.values()), "문항 목표")

20 문항 목표


In [103]:
import numpy as np

V = np.array(embeddings.embed_documents([c.page_content for c in all_chunks]))
print(V.shape)

(275, 1536)


In [143]:
def having_all(*words, n):
    return [c for c in all_chunks if all(w in c.page_content for w in words)][:n]

def having_any(*words, n):
    return [c for c in all_chunks if any(w in c.page_content for w in words)][:n]

step = max(1, len(all_chunks) // (TARGET["단일홉"] * OVERSAMPLE))
single = all_chunks[::step][:TARGET["단일홉"] * OVERSAMPLE]
counting = having_all("①", "②", n=TARGET["집계"] * (OVERSAMPLE+1))
condition = having_any("위해", "구분", n=TARGET["조건"] * (OVERSAMPLE+2))
step = having_any("프로세스", "단계", n=TARGET["절차"] * OVERSAMPLE)
print({"단일홉": len(single), "집계": len(counting), "조건": len(condition), "절차": len(step)})

{'단일홉': 12, '집계': 9, '조건': 8, '절차': 4}


In [144]:
pairs = []
for i in range(0, len(all_chunks), max(1, len(all_chunks) // (TARGET["멀티홉"] * OVERSAMPLE))):
    order = np.argsort(-(V @ V[i]))
    j = next(
        (
            int(k)
            for k in order[1:275]
            if (
                all_chunks[int(k)].metadata["source"]
                != all_chunks[i].metadata["source"]
                or
                all_chunks[int(k)].metadata["page"]
                != all_chunks[i].metadata["page"]
            )
        ),
        None
    )
    if j is not None:
        pairs.append((all_chunks[i], all_chunks[j]))
pairs = pairs[:TARGET["멀티홉"] * OVERSAMPLE]
print(len(pairs), "쌍", [(a.metadata["page"], b.metadata["page"]) for a, b in pairs[:3]])

8 쌍 [(0, 34), (19, 20), (21, 18)]


In [145]:
from langchain_core.prompts import ChatPromptTemplate
from typing import Literal
from pydantic import BaseModel, Field

# todo : draft에 필요한 항목들
class Draft(BaseModel):
    question: str = Field(description="질문만 읽어도 무엇을 묻는지 특정되어야 한다. 지시대명사를 쓰지 않는다")
    answer: str
    quote: str = Field(description="근거에서 그대로 복사한 한 문장. 요약하거나 고치지 않는다")
    difficulty: Literal["easy", "medium", "hard"]

writer = ChatPromptTemplate.from_messages([
    ("system", "현대홈쇼핑 협력사용 가이드 문서를 근거로 질문과 답을 만든다. 근거에 없는 사실을 쓰지 않는다.\n{guide}"),
    ("human", "근거\n{context}"),
]) | ChatOpenAI(model=MODEL).with_structured_output(Draft)

In [155]:
GUIDES = {
    "단일홉": "근거 한 곳만 보면 답할 수 있는 질문을 만든다.",
    "멀티홉": "두 근거를 모두 봐야 답할 수 있는 질문을 만든다. 한쪽만 보면 답이 나오지 않아야 한다.",
    "집계": "근거에 열거된 항목이 몇 개인지 세거나 나열하는 질문을 만든다. 답에는 개수와 목록을 함께 쓴다.",
    "조건": "무엇을 하기 위해 필요한 것을 물어보는 질문을 만든다. 혹은 구분되는 분류 항목들에 따른 서로 다른 조건을 물어보는 질문을 만든다.",
    "절차": "무엇을 진행하기 위해 필요한 절차를 물어보는 질문을 만든다. 답은 단계별로 정해진 순서대로 쓴다.",
}

In [156]:
jobs = ([("단일홉", [c]) for c in single] + [("멀티홉", list(p)) for p in pairs]
        + [("집계", [c]) for c in counting] + [("조건", [c]) for c in condition]
        + [("절차", [c]) for c in step])
print(len(jobs), "건")

41 건


In [157]:
drafts = writer.batch([{"guide": GUIDES[t], "context": "\n\n".join(c.page_content for c in cs)}
                       for t, cs in jobs], config={"callbacks": [usage]})
print(len(drafts), "초안")

41 초안


In [158]:
for d in drafts :
    print (d)

question='현대홈쇼핑 협력사운영통합안내서의 발행 시점은 언제인가요?' answer='2024.06입니다.' quote='2024.06' difficulty='easy'
question='주말과 공휴일은 출고기준일 산정에 포함되는가?' answer='포함되지 않는다.' quote='○주말/공휴일은출고기준일산정에미포함' difficulty='easy'
question='판매수수료기반 CPS(Cost per sale) 제휴 담당자 연락처는 무엇인가?' answer='yunsh_@hyundaihmall.com' quote='**▷담당자연락처yunsh_@hyundaihmall.com**' difficulty='easy'
question='플랫폼행사는 Hmall의 어느 영역에 노출됩니까?' answer='플랫폼행사는 Hmall의 PC · 모바일 메인영역에 노출됩니다.' quote='플랫폼행사는Hmall의PC · 모바일메인영역에노출됩니다.' difficulty='easy'
question='검색 광고 영역에서 A_검색창문구의 단가와 노출기간은 얼마인가요?' answer='단가는 180만원이고 노출기간은 1일입니다.' quote='검색창 A_검색창문구 1개 180만원 1일 1개' difficulty='easy'
question='표준거래계약서에 서명할 때 첨부해야 하는 서류 3가지는 무엇인가요?' answer='보증보험, 통장사본, 사업자등록증입니다.' quote='①보증보험 ②통장사본 ③사업자등록증 을해당계약서에첨부하고 기업범용공인인증서 ( 사업자용) 로서명합니다.' difficulty='easy'
question='하루 시청자 수가 85만명인 배포용 데이터 영역의 가치는 카탈로그 의류만 판매 기준 연 매출액이 얼마인가?' answer='100억' quote='85만명' difficulty='easy'
question='당도를 소구하는 경우 추가로 필요한 제출 서류는 무엇인가요?' answer='브릭스측정자료(당도소구시필요)입니다.' quote='브릭스측정자료(당도

In [159]:
# 검수1. 인용 확인 (짧은 인용 제거)
def squash(s):
    return "".join(s.split())

stage1, dropped = [], []
for (kind, cs), d in zip(jobs, drafts):
    source = squash("\n".join(c.page_content for c in cs))
    if len(squash(d.quote)) > 10 and squash(d.quote) in source:
        stage1.append((kind, cs, d))
    else:
        dropped.append((kind, "인용 불일치", d.question, d.quote, d.answer))
print(f"통과 {len(stage1)}  탈락 {len(dropped)}")

for row in dropped:
    print("  ", row)

통과 31  탈락 10
   ('단일홉', '인용 불일치', '현대홈쇼핑 협력사운영통합안내서의 발행 시점은 언제인가요?', '2024.06', '2024.06입니다.')
   ('단일홉', '인용 불일치', '하루 시청자 수가 85만명인 배포용 데이터 영역의 가치는 카탈로그 의류만 판매 기준 연 매출액이 얼마인가?', '85만명', '100억')
   ('단일홉', '인용 불일치', '린넨으로 소구하려면 혼용률 세부감별 결과에서 무엇이 확인되어야 하나요?', '* 린넨: 마세부감별결과‘아마’ 확인', '혼용률 세부감별 결과에서 ‘아마’가 확인되어야 합니다.')
   ('멀티홉', '인용 불일치', '포장 관련 제출물과 사용 금지 사항을 함께 고려할 때, 고객이 받았을 때 보이는 모든 한글표시사항 시안으로 제출해야 하는 항목 두 가지와 샘플 포장에 사용 가능한 테이프 및 사용 금지 테이프는 무엇인가요?', '- 개별포, 파우치, 개별상자, 단상자, 테이프, 구성별합포장사진시안, 바틀/스푼등추가구성사용시전체 시안', "시안으로 제출해야 하는 항목으로는 개별포와 단상자가 포함됩니다. 샘플 포장에는 종이 무지테이프를 사용할 수 있고, '개봉후반품불가' 테이프는 사용이 금지됩니다.")
   ('멀티홉', '인용 불일치', '메탈섬유가 포함된 원단을 100% 소재로 소구할 수 있는지와, 혼용률 시험에서 100% 소재에 적용되는 오차 기준은 무엇인지 설명하시오.', '* 메탈섬유: ‘금속성섬유(지정외)’ 확인후소구\n100% 소재는오차없음', '메탈섬유는 혼용률 항목에서 ‘금속성섬유(지정외)’ 확인 후 소구해야 하므로, 제시된 근거만으로는 일반적인 천연섬유 100%처럼 단독으로 100% 소재 소구가 가능하다고 볼 수 없습니다. 또한 혼용률 이화학 공통 기준에서는 100% 소재는 오차가 없다고 되어 있습니다.')
   ('집계', '인용 불일치', '방송상품QA 프로세스에 열거된 단계는 모두 몇 개인가요? 방송상품QA 프로세스의 단계 목록을 순서대로 쓰세요.', '①QA 의

In [160]:
# 검수2. 자기완결성
class SelfContained(BaseModel):
    ok: bool = Field(description="질문만 읽고 무엇을 묻는지 특정되면 True")
    reason: str


checker = ChatPromptTemplate.from_messages([
    ("system", "현대홈쇼핑 협력사용 가이드 문서에 대한 질문이다. 이 질문을 문서 전체를 아는 사람에게 "
               "단독으로 던졌을 때 무엇을 묻는지 특정되는지 판단한다. "
               "주어나 대상이 빠져 있거나 '이 경우' 같은 지시대명사에 기대면 False 다."),
    ("human", "{question}"),
]) | ChatOpenAI(model=JUDGE).with_structured_output(SelfContained)


In [161]:
verdicts = checker.batch([{"question": d.question} for _, _, d in stage1], config={"callbacks": [usage]})
print(f"판정 {len(verdicts)}건 중 통과 {sum(v.ok for v in verdicts)}건")

판정 31건 중 통과 30건


In [162]:
stage2 = []
for (kind, cs, d), v in zip(stage1, verdicts):
    if v.ok:
        stage2.append((kind, cs, d))
    else:
        dropped.append((kind, "자기완결성 없음", d.question))
        print("  탈락", d.question, "|", v.reason)
print(f"통과 {len(stage2)}")

  탈락 두 소재 유형 모두에서 공통으로 요구되는 해상도와 파일포맷은 무엇이며, 안내사항에서는 각 유형별로 어떤 추가 조건이 다른가? | '두 소재 유형'과 '각 유형'이 어떤 소재 유형을 가리키는지 질문 안에 명시되어 있지 않아, 문서 전체를 아는 사람에게 단독으로 던져도 비교 대상이 특정되지 않습니다.
통과 30


In [163]:
# 수동검수
for i, (kind, cs, d) in enumerate(stage2):
    print(f"{i:2d} [{kind}] {d.question}")
    print(f"     답 {d.answer}")
    print(f"     근거 doc.{sorted({c.metadata['source'] for c in cs})} p.{sorted({c.metadata['page']+1 for c in cs})}  {d.quote[:55]}")

 0 [단일홉] 주말과 공휴일은 출고기준일 산정에 포함되는가?
     답 포함되지 않는다.
     근거 doc.['1. 협력사 운영 통합 안내서 (p.23).pdf'] p.[16]  ○주말/공휴일은출고기준일산정에미포함
 1 [단일홉] 판매수수료기반 CPS(Cost per sale) 제휴 담당자 연락처는 무엇인가?
     답 yunsh_@hyundaihmall.com
     근거 doc.['1. 협력사 운영 통합 안내서 (p.23).pdf'] p.[22]  **▷담당자연락처yunsh_@hyundaihmall.com**
 2 [단일홉] 플랫폼행사는 Hmall의 어느 영역에 노출됩니까?
     답 플랫폼행사는 Hmall의 PC · 모바일 메인영역에 노출됩니다.
     근거 doc.['2. Hmall 소개서 (p.30).pdf'] p.[20]  플랫폼행사는Hmall의PC · 모바일메인영역에노출됩니다.
 3 [단일홉] 검색 광고 영역에서 A_검색창문구의 단가와 노출기간은 얼마인가요?
     답 단가는 180만원이고 노출기간은 1일입니다.
     근거 doc.['4. 광고 상품 소개서 (p.24).pdf'] p.[12]  검색창 A_검색창문구 1개 180만원 1일 1개
 4 [단일홉] 표준거래계약서에 서명할 때 첨부해야 하는 서류 3가지는 무엇인가요?
     답 보증보험, 통장사본, 사업자등록증입니다.
     근거 doc.['6. 신규 협력사 입점 절차 안내 (p.21).pdf'] p.[6]  ①보증보험 ②통장사본 ③사업자등록증 을해당계약서에첨부하고 기업범용공인인증서 ( 사업자용) 로서명합니
 5 [단일홉] 당도를 소구하는 경우 추가로 필요한 제출 서류는 무엇인가요?
     답 브릭스측정자료(당도소구시필요)입니다.
     근거 doc.['8. QA가이드 (p.21).pdf'] p.[10]  브릭스측정자료(당도소구시필요)
 6 [단일홉] 액체성상상품은 하루 최고기온이 몇 도 이하인 혹한기에 출고 전 상품 성상 점검이 필수인가요?
     답 하루 최고

절차 : 계속 동일한것만 여러개 가져옴 -> 가져오는 조건을 바꾸거나, 절차형을 알아볼 수 있게 개선 필요

In [164]:
selected, seen = [], {k: 0 for k in TARGET}
for i, (kind, cs, d) in enumerate(stage2):
    if i in REJECT or seen[kind] >= TARGET[kind]:
        continue
    seen[kind] += 1
    selected.append((kind, cs, d))
print({k: v for k, v in seen.items() if k != "무답변"}, " 합계", len(selected))

{'단일홉': 6, '멀티홉': 4, '집계': 3, '조건': 2, '절차': 2}  합계 17


In [165]:
for kind, want in TARGET.items():
    if kind != "무답변" and seen[kind] < want:
        print(f"  {kind} 이 {want - seen[kind]}개 모자란다")

In [168]:
UNANSWERABLE = [("협력사 시스템 사용자 추가는 누구나 할 수 있나요?", "대표계정"),
                ("쇼라의 누적매출액은 얼마인가요?", "5800억"),
                ("협력사 휴무일 신청은 어디서 하나요?", "VD106")]

In [180]:
whole = squash("".join(c.page_content for c in all_chunks))
for q, key in UNANSWERABLE:
    print(f"{key:8} 코퍼스 등장 {whole.count(key)}회   {q}")

대표계정     코퍼스 등장 0회   협력사 시스템 사용자 추가는 누구나 할 수 있나요?
5800억    코퍼스 등장 0회   쇼라의 누적매출액은 얼마인가요?
VD106    코퍼스 등장 0회   협력사 휴무일 신청은 어디서 하나요?


In [170]:
# 스키마 저장
records = [{"id": f"lease-{i:03d}", "type": kind, "question": d.question,
            "reference_answer": d.answer, "answerable": True,
            "evidence_pages": sorted({c.metadata["page"] for c in cs}),
            "evidence_quotes": [d.quote], "difficulty": d.difficulty}
           for i, (kind, cs, d) in enumerate(selected, 1)]
print(len(records), "문항")

17 문항


In [171]:
from collections import Counter

records += [{"id": f"lease-{j:03d}", "type": "무답변", "question": q,
             "reference_answer": "문서에 없음", "answerable": False,
             "evidence_pages": [], "evidence_quotes": [], "difficulty": "medium"}
            for j, (q, _) in enumerate(UNANSWERABLE, len(records) + 1)]
print(len(records), "문항", Counter(r["type"] for r in records))

20 문항 Counter({'단일홉': 6, '멀티홉': 4, '집계': 3, '무답변': 3, '조건': 2, '절차': 2})


In [173]:
import json
from pathlib import Path

print(json.dumps(records[0], ensure_ascii=False, indent=1))

Path(GOLDEN).parent.mkdir(parents=True, exist_ok=True)
Path(GOLDEN).write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in records) + "\n", encoding="utf-8")

{
 "id": "lease-001",
 "type": "단일홉",
 "question": "주말과 공휴일은 출고기준일 산정에 포함되는가?",
 "reference_answer": "포함되지 않는다.",
 "answerable": true,
 "evidence_pages": [
  15
 ],
 "evidence_quotes": [
  "○주말/공휴일은출고기준일산정에미포함"
 ],
 "difficulty": "easy"
}


6569

In [181]:
#근거
def is_evidence(chunk, record):
    text = squash(chunk.page_content)
    return any(squash(q) in text for q in record["evidence_quotes"])


print(sum(is_evidence(c, records[0]) for c in all_chunks), "청크가 첫 문항의 근거다")

1 청크가 첫 문항의 근거다


In [183]:
from langchain_core.output_parsers import StrOutputParser

probe = [r for r in records if r["answerable"]][:5]
ask = ChatPromptTemplate.from_messages([("system", "근거만으로 두 문장 이내로 답한다."),
                                        ("human", "근거\n{ctx}\n\n질문\n{q}")])
ctx = {r["id"]: " ".join(c.page_content for c in all_chunks if is_evidence(c, r)) for r in probe}
reference = [r["reference_answer"] for r in probe]
print(len(probe), "문항")

5 문항


In [186]:
answers = {}
for label, name in [("참조를 쓴 모델", MODEL), ("평가 대상 모델", LOWER)]:
    answers[label] = (ask | ChatOpenAI(model=name) | StrOutputParser()).batch(
        [{"ctx": ctx[r["id"]], "q": r["question"]} for r in probe], config={"callbacks": [usage]})
    print(f"{label:14} {answers[label][3]}")

참조를 쓴 모델       A_검색창문구의 단가는 180만원이며, 노출기간은 1일입니다.  
근거 표의 “검색창 A_검색창문구 1개 180만원 1일 1개”에 명시되어 있습니다.
평가 대상 모델       A_검색창문구의 단가는 180만원이고, 노출기간은 1일입니다.


In [187]:
R = np.array(embeddings.embed_documents(reference))
for label, outs in answers.items():
    A = np.array(embeddings.embed_documents(outs))
    print(f"{label:14} 참조와의 평균 코사인 {float((A * R).sum(axis=1).mean()):.4f}")

참조를 쓴 모델       참조와의 평균 코사인 0.6138
평가 대상 모델       참조와의 평균 코사인 0.6875
